# LiteRT 온디바이스 이미지 분류: 최종 리포트 (Final Report)

## 1. 프로젝트 개요 (Project Overview)
본 프로젝트는 **LiteRT (구 TensorFlow Lite)** 를 사용하여 **MobileNetV2** 이미지 분류 모델을 온디바이스 환경에 최적화하는 과정을 다룹니다.
다음 세 가지 핵심 양자화 전략을 비교 분석합니다:

1.  **FP32 (Base):** 표준 32비트 부동소수점 변환 (기준 모델).
2.  **INT8 (Legacy Static):** ONNX를 경유하여 Representative Dataset을 적용한 Full Integer Quantization 방식.
3.  **INT8 (ai-edge Hybrid):** `ai-edge-torch`의 **'SavedModel Intercept'** 기법을 활용한 새로운 Full Integer Quantization 방식.

**목표:** 라이브러리의 한계를 극복하고, 모델 크기와 추론 속도(Latency) 간의 최적의 균형을 찾습니다.

In [1]:
import sys
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

# 소스 코드 패키지 경로 추가
sys.path.append(os.path.abspath(".."))

from src import run_inference, run_comparison, load_labels, convert_aiedge_int8

# 그래프 스타일 설정
plt.style.use('ggplot')

2026-01-18 22:19:51.550671: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/home/dhkim/workspace/study/lite_rt/.venv/lib/python3.10/site-packages/torch/distributed/distributed_c10d.py:351: UserWarning: Device capability of jax unspecified, assuming `cpu` and `cuda`. Please specify it via the `devices` argument of `register_backend`.
  warnings.warn(
/home/dhkim/workspace/study/lite_rt/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. 모델 목록 확인 (Model Inventory)
변환이 완료된 모델 파일들을 확인합니다.

In [2]:
!ls -lh ../models/*.tflite

/home/dhkim/.local/share/uv/python/cpython-3.10.19-linux-x86_64-gnu/lib/python3.10/pty.py:89: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid, fd = os.forkpty()


-rw-rw-r-- 1 dhkim dhkim 3.9M  1월 18 12:53 ../models/mobilenet_v2_aiedge_int8.tflite
-rw-rw-r-- 1 dhkim dhkim 3.9M  1월 18 12:21 ../models/mobilenet_v2_legacy_int8.tflite
-rw-rw-r-- 1 dhkim dhkim  14M  1월 17 00:32 ../models/mobilenet_v2.tflite


## 3. 도전 과제 및 해결책: ai-edge-torch INT8

### 문제점 (The Problem)
`ai_edge_torch.convert`를 사용하여 직접 INT8 양자화를 시도했을 때, 생성된 모델의 크기가 줄어들지 않거나(13MB 유지), 강제로 Static Quantization을 적용하려 하면 `stablehlo.uniform_dequantize` 오퍼레이션 호환성 문제로 변환에 실패했습니다.

### 해결책: "SavedModel Intercept" 전략
우리는 이 문제를 해결하기 위해 다음과 같은 하이브리드 방식을 고안했습니다:

1.  **고품질 그래프 추출:** `ai-edge-torch`를 사용하여 PyTorch 모델을 **TensorFlow SavedModel** 형태로 내보냅니다. 이 단계에서는 TFLite 변환을 수행하지 않고, 우수한 PT2E Lowering 기능만 활용합니다.
2.  **안정적인 양자화:** 추출된 SavedModel에 대해 표준 `tf.lite.TFLiteConverter`를 적용합니다. 이를 통해 TensorFlow의 성숙한 양자화 파이프라인을 사용하여 호환성 문제 없이 완벽한 INT8 모델을 생성할 수 있었습니다.

**결과:** 13MB였던 모델을 **3.8MB**로 성공적으로 경량화했습니다.

## 4. 방법론 비교: ONNX Route vs ai-edge Hybrid
두 가지 성공적인 변환 방식의 기술적 차이점과 특징을 비교합니다.

| 비교 항목 | **Legacy Route (ONNX)** | **Modern Hybrid Route (ai-edge)** |
| :--- | :--- | :--- |
| **변환 파이프라인** | PyTorch → **ONNX** → TF SavedModel → TFLite | PyTorch → **StableHLO** → TF SavedModel → TFLite |
| **핵심 기술** | **ONNX Intermediate Representation** | **Google StableHLO (XLA)** |
| **특징** | • 오래 검증된 방식 (호환성 높음)<br>• 외부 라이브러리(`onnx`, `onnx2tf`) 의존성 높음<br>• 변환 단계가 많아 복잡함 | • **Google 공식 권장 경로** (최신 기술)<br>• 불필요한 제3 포맷(ONNX) 변환 없음<br>• 파이프라인이 더 간결하고 미래 지향적 |
| **우리의 전략** | 표준적인 절차를 그대로 따름 | `ai-edge-torch`의 양자화 버그를 우회하기 위해 **"그래프 추출"과 "양자화" 단계를 분리**하는 하이브리드 기법 적용 |

**분석:**
두 방식 모두 최종적으로는 동일한 **TensorFlow SavedModel** 상태에서 **표준 TFLiteConverter**를 통과하게 되므로, 최종 모델의 성능(크기 3.8MB, Latency)은 거의 동일합니다. 하지만 **유지보수와 미래 호환성 측면에서 ai-edge Hybrid 방식이 더 우수**합니다.

## 5. 벤치마크 결과 (Benchmarking Results)
FP32 베이스라인 모델과 두 가지 성공적인 INT8 모델의 성능을 비교합니다.

In [3]:
# 벤치마크 실행 및 비교
results = run_comparison()

# 데이터프레임으로 결과 표시
df = pd.DataFrame(results)
df

Searching for models in: /home/dhkim/workspace/study/lite_rt/models
Found 3 models. Starting benchmark...

Benchmarking /home/dhkim/workspace/study/lite_rt/models/mobilenet_v2.tflite...
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
Benchmarking /home/dhkim/workspace/study/lite_rt/models/mobilenet_v2_aiedge_int8.tflite...
Benchmarking /home/dhkim/workspace/study/lite_rt/models/mobilenet_v2_legacy_int8.tflite...

COMPARISON RESULTS
+---------------------------------+-------------+--------------------+------------+------------+-----------------+
| Model                           |   Size (MB) |   Avg Latency (ms) |   Min (ms) |   Max (ms) |   Peak Mem (MB) |
+=================================+=============+====================+============+============+=================+
| mobilenet_v2.tflite             |       13.43 |               5.56 |       5.37 |       7.18 |         1297.16 |
+---------------------------------+-------------+--------------------+------------+------------+

,file,size_mb,avg_ms,min_ms,max_ms,std_ms,mem_peak_mb
0,mobilenet_v2.tflite,13.426849,5.558968,5.370378,7.177830,0.291854,1297.15625
1,mobilenet_v2_aiedge_int8.tflite,3.828400,6.900692,6.884098,7.021427,0.019308,1297.28125
2,mobilenet_v2_legacy_int8.tflite,3.822128,6.833973,6.821156,6.975651,0.024127,1297.28125


### 성능 분석 (Analysis)
- **모델 크기 (Size):** 두 INT8 모델 모두 **약 72%의 압축률** (13.4MB -> 3.8MB)을 달성했습니다.
- **지연 시간 (Latency):** x86 CPU 환경에서는 INT8 모델이 FP32 모델보다 느리게 측정되는데(~7ms vs ~5ms), 이는 CPU 최적화 차이 때문입니다.

## 6. 추가 검증: 실제 안드로이드 기기 벤치마크 (Extreme Performance)
x86 CPU에서는 볼 수 없었던 INT8의 진가를 실제 **안드로이드 기기 (ARM64)**에서 확인했습니다.

| 모델 (Model) | 모바일 CPU (ms) | 모바일 GPU (ms) |
| :--- | :--- | :--- |
| **FP32** | 68.52ms | **15.88ms** |
| **INT8 (ai-edge Hybrid)** | **39.95ms** | 17.57ms |

**핵심 요약:**
- **모바일 CPU**에서 INT8 모델은 FP32 대비 **약 1.7배 빠른 성능**을 보여주었습니다.
- 양자화를 통해 **용량(72% 감소)**과 **속도(1.7배 향상)**를 동시에 잡을 수 있음을 실제 기기 테스트를 통해 입증했습니다.

## 7. 결론 (Conclusion)
`ai-edge-torch`의 직접 변환 기능에 버그가 있더라도, **SavedModel Intercept** 전략을 사용하면 복잡한 ONNX 변환 과정을 거치지 않고도 PyTorch 모델을 효과적으로 INT8로 양자화할 수 있음을 입증했습니다. 이는 구글의 최신 기술 스택을 유지하면서도 안정적인 실 서비스 배포가 가능한 강력한 방법론입니다.